# Monthly report — January 2026

Exploration notebook of the data team. Same numbers as `scripts/monthly_report.py`.
Run all cells from the top.

In [ ]:
import os
import pandas as pd

In [ ]:
DATA = "../data"
OUT = "../out"
MONTH = "2026_01"

In [ ]:
df = pd.read_csv(DATA + "/orders_" + MONTH + ".csv", dtype={"promo_code": str})
prods = pd.read_csv(DATA + "/products.csv")
custs = pd.read_csv(DATA + "/customers_pii.csv")
df.head()

In [ ]:
# df.describe()
# df[df.quantity >= 50].sort_values("quantity").tail(20)
# df.groupby("country").size()
# custs[custs.tier == "gold"].head()

In [ ]:
def vol(q):
    if q >= 50:
        return 0.10
    elif q >= 10:
        return 0.05
    return 0.0

cat = {}
for i, r in prods.iterrows():
    cat[r["product_id"]] = (r["category"], r["name"])
tier = {}
for i, r in custs.iterrows():
    tier[r["customer_id"]] = r["tier"]

In [ ]:
data = {}
for i, r in df.iterrows():
    oid = r["order_id"]
    if oid not in data:
        data[oid] = {"customer": r["customer_id"], "country": r["country"], "lines": []}
    data[oid]["lines"].append((r["product_id"], int(r["quantity"]), float(r["unit_price"])))
nb = len(data)
rows = []
nb

In [ ]:
# VAT per country
rates = {"FR": 0.20, "DE": 0.19, "ES": 0.21, "BE": 0.21}
res = {}
for country in ["FR", "DE", "ES", "BE"]:
    net = 0.0
    ship_total = 0.0
    vat = 0.0
    n = 0
    for oid in data:
        o = data[oid]
        if o["country"] != country:
            continue
        s = 0.0
        for p, q, up in o["lines"]:
            s = s + q * up * (1 - vol(q))
        if tier[o["customer"]] == "gold":
            s = s * 0.97
        ship = 0.0
        if s < 100:
            ship = 6.90
        net += s
        ship_total += ship
        vat += (s + ship) * rates[country]
        n += 1
    res[country] = (net, ship_total, vat, n)
    rows.append(["country", country, "orders", str(n)])
    rows.append(["country", country, "net", "%.2f" % net])
    rows.append(["country", country, "shipping", "%.2f" % ship_total])
    rows.append(["country", country, "vat", "%.2f" % vat])
    rows.append(["country", country, "gross", "%.2f" % (net + ship_total + vat)])
res

In [ ]:
bycat = {}
for oid in data:
    for p, q, up in data[oid]["lines"]:
        c = cat[p][0]
        bycat[c] = bycat.get(c, 0.0) + q * up * (1 - vol(q))
for c in sorted(bycat):
    rows.append(["category", c, "net", "%.2f" % bycat[c]])
bycat

In [ ]:
tmp = {}
for oid in data:
    for p, q, up in data[oid]["lines"]:
        tmp[p] = tmp.get(p, 0.0) + q * up * (1 - vol(q))
best = sorted(tmp.items(), key=lambda x: (-round(x[1], 2), x[0]))[:10]
for k, (p, v) in enumerate(best, start=1):
    rows.append(["top_product", "%02d" % k, p + " " + cat[p][1], "%.2f" % v])
best

In [ ]:
gross = 0.0
for country in ["FR", "DE", "ES", "BE"]:
    gross = gross + res[country][0]
    gross = gross + res[country][1]
    gross = gross + res[country][2]
big = 0
for oid in data:
    s = 0.0
    for p, q, up in data[oid]["lines"]:
        s = s + q * up * (1 - vol(q))
    if tier[data[oid]["customer"]] == "gold":
        s = s * 0.97
    if s >= 100:
        big += 1
gross, big

In [ ]:
# careful: data is now the list of customers
data = sorted(set(df["customer_id"]))
g = len([c for c in data if tier[c] == "gold"])
rows.append(["kpi", "orders", "count", str(nb)])
rows.append(["kpi", "gross_revenue", "eur", "%.2f" % gross])
rows.append(["kpi", "average_basket", "eur", "%.2f" % (gross / nb)])
rows.append(["kpi", "orders_over_100", "count", str(big)])
rows.append(["kpi", "customers", "count", str(len(data))])
rows.append(["kpi", "gold_share", "percent", "%.2f" % (100.0 * g / len(data))])

In [ ]:
os.makedirs(OUT, exist_ok=True)
with open(OUT + "/report_" + MONTH + ".csv", "w", newline="") as f:
    f.write("section,key,label,value\n")
    for r in rows:
        f.write(",".join(r) + "\n")
len(rows)

In [ ]:
print("Monthly report", MONTH)
print("orders:", nb, "- gross revenue:", "%.2f" % gross, "- average basket:", "%.2f" % (gross / nb))
print("gold customers: %.2f%%" % (100.0 * g / len(data)))